# 08 - Fetch the raw weather
This file fetches the raw weather data from the API, and saves it as one JSON file per tournament in raw/open_meteo/archive_v2/


## Install the pinned libraries

In [ ]:

REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Load the edition list

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
editions = bq.query(
    f"SELECT * FROM `{PROJECT_ID}.tennis_clean.tournament_editions` ORDER BY tourney_id",
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000),
).to_dataframe()
for c in ["start_date", "end_date"]:
    editions[c] = pd.to_datetime(editions[c])
print(len(editions), "editions loaded")

## The fetch function
Each response is saved exactly as returned, with the request and fetch time stored beside it.
If a file already exists it is only skipped when it was fetched with exactly the parameters now wanted.
The model is pinned to ERA5.

In [ ]:
import json, time, hashlib
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = "era5"
HOURLY = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "surface_pressure",
          "precipitation", "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cloud_cover"]
RAW_PREFIX = f"raw/open_meteo/archive_v2/model={MODEL}"

def fetch_edition(row, model=MODEL, store=True):
    """Fetch one edition's weather. With store=True: skip if already saved, else save write-once."""
    params = {
        "latitude": round(float(row.latitude), 4), "longitude": round(float(row.longitude), 4),
        "start_date": row.start_date.strftime("%Y-%m-%d"), "end_date": row.end_date.strftime("%Y-%m-%d"),
        "hourly": ",".join(HOURLY), "timezone": "auto",
    }
    if model:
        params["models"] = model

    blob = bucket.blob(f"{RAW_PREFIX}/{row.tourney_id}.json")
    if store and blob.exists():
        stored = json.loads(blob.download_as_text())
        if stored["request"]["params"] != params:
            raise RuntimeError(f"{row.tourney_id}: stored file was fetched with different parameters")
        return "already stored", None

    for attempt in range(5):                       # retry on busy or temporary server errors
        r = requests.get(ARCHIVE_URL, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * 2 ** attempt)           # wait 2, 4, 8, 16, 32 seconds
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"Gave up on {row.tourney_id} (HTTP {r.status_code})")

    record = {
        "tourney_id": row.tourney_id,
        "request": {"url": ARCHIVE_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),                      # exactly as returned
    }
    if store:
        blob.upload_from_string(json.dumps(record), content_type="application/json",
                                if_generation_match=0)
    return "fetched", record

## Test that the model setting works (nothing is saved if different model used)
identical should be 'FALSE'

In [ ]:
row = next(editions[editions["start_date"].dt.strftime("%Y-%m") == "2024-06"].itertuples())
_, pinned = fetch_edition(row, model="era5", store=False)
time.sleep(1)
_, default = fetch_edition(row, model=None, store=False)

a = pinned["response"]["hourly"]["temperature_2m"]
b = default["response"]["hourly"]["temperature_2m"]
print(row.place, row.start_date.date())
print("era5    first 6 hours:", a[:6])
print("default first 6 hours:", b[:6])
print("identical:", a == b)

## Fetch every edition
If `failed` is not empty, run this cell once more.

In [ ]:
failed, counts = [], {"fetched": 0, "already stored": 0}
for i, row in enumerate(editions.itertuples(), 1):
    try:
        status, _ = fetch_edition(row)
        counts[status] += 1
        if status == "fetched":
            time.sleep(0.5)                        # about 2 requests a second at most
    except Exception as e:
        failed.append((row.tourney_id, repr(e)[:150]))
    if i % 100 == 0:
        print(i, counts, "failed:", len(failed))

print("Done:", counts, "| failed:", len(failed))
failed[:5]

## Check what is stored


In [ ]:
entries = []
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/"):
    data = b.download_as_bytes()
    rec = json.loads(data)
    hourly = rec["response"]["hourly"]
    p = rec["request"]["params"]
    entries.append({
        "file": b.name, "tourney_id": rec["tourney_id"], "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
        "start_date": p["start_date"], "end_date": p["end_date"],
        "n_hours": len(hourly["time"]),
        "null_values": sum(v is None for k, vals in hourly.items() if k != "time" for v in vals),
        "fetched_at_utc": rec["fetched_at_utc"],
    })

wanted = editions[["tourney_id", "start_date", "end_date"]].copy()
wanted["start_date"] = wanted["start_date"].dt.strftime("%Y-%m-%d")
wanted["end_date"] = wanted["end_date"].dt.strftime("%Y-%m-%d")
check = wanted.merge(pd.DataFrame(entries), on="tourney_id", how="left", suffixes=("_wanted", "_stored"))
check["expected_hours"] = ((pd.to_datetime(check["end_date_wanted"]) - pd.to_datetime(check["start_date_wanted"])).dt.days + 1) * 24

print("files stored:", len(entries), "| editions with no file:", int(check["file"].isna().sum()))
print("files with a different window than wanted:",
      int(((check["start_date_stored"] != check["start_date_wanted"]) | (check["end_date_stored"] != check["end_date_wanted"])).sum()))
print("files whose hour count differs from the window by more than 2:", int(((check["n_hours"] - check["expected_hours"]).abs() > 2).sum()))
print("total null values:", int(check["null_values"].sum()))
print("total size (MB):", round(check["bytes"].sum() / 1e6, 1))

## Look at one stored file (timezone, units, first hours)

In [ ]:
rec = json.loads(bucket.blob(f"{RAW_PREFIX}/{editions.iloc[0].tourney_id}.json").download_as_text())
resp = rec["response"]
print({k: resp.get(k) for k in ["latitude", "longitude", "elevation", "timezone", "hourly_units"]})
pd.DataFrame(resp["hourly"]).head(6)

## Save the manifest

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_archive_v2_{MODEL}_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": ARCHIVE_URL, "model": MODEL, "hourly_variables": HOURLY,
                "window_rule": "start = tourney_date; end = latest possible match day + 1 (tennis_clean.tournament_editions)",
                "files": entries}, indent=1),
    content_type="application/json")
print("Saved", manifest_path, "with", len(entries), "files")